清理腳本：把「更新資料」(intakes + outcomes) 清成模型的 out-of-time test set
=================================================================================
依據 diagnose_new_system_data.py 的證據與已確認的判讀決策：

  決策 1（母體定義）：Row-level by outcome_status —— 保留所有 source_name，
      逐筆排除「非認養/正常收容流程」的 outcome_status。
      證據：沒有任何 source_name 是放回主導（最高 5%），所以不整類排除
      Community Animal - Program。
  決策 2（配不到 intake 的列）：排除 —— 配不到 intake 就無法重算單次住院天數，
      無法產生可信的 is_long_stay 標籤，直接剔除。

本腳本產出的 CSV 欄位與 data/processed/df_full_merged.csv 完全對齊（26 欄），
可直接當成 03_modeling 的 out-of-time test fold 餵進既有 pipeline。

length_of_stay_days 用「重算」的單次住院（outcome_date - 配到的 intake.source_date），
絕不使用原檔被灌水的 days_in_shelter。

用法：python3 build_oot_test_set.py

In [10]:
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

INTAKES_PATH = "../../data/new_system_data/austin_animal_intakes.csv"
OUTCOMES_PATH = "../../data/new_system_data/austin_animal_outcomes.csv"
TRAIN_SCHEMA_PATH = "../../data/processed/df_full_merged.csv"
OUTPUT_PATH = "../../data/processed/oot_test_set.csv"

LONG_STAY_THRESHOLD = 30
DROP_UNMATCHED = True  # 決策 2：配不到 intake 的 outcome 直接排除

# ---------------------------------------------------------------------------
# 可編輯設定（判讀決策都集中在這裡，看完輸出可隨時調整）
# ---------------------------------------------------------------------------

# 決策 1：保留哪些 outcome_status（認養/領回/轉出/安樂死＝正常收容流程），
#         並對映回舊資料的 outcome_type 詞彙（讓 OUTCOME_CATEGORY_MAP 能套用）。
#         不在這張表裡的 status 一律視為「非母體」被排除
#         （Return to Habitat / Released / DOA* / Unassisted Death* /
#          Escaped / Stolen / Unresolved File / Bite Quarantine 等）。
OUTCOME_STATUS_TO_TYPE = {
    "Adopted":             "Adoption",
    "Adopted Offsite":     "Adoption",
    "Adopted Unaltered":   "Adoption",
    "Transfer Out":        "Transfer",
    "Reclaimed":           "Return to Owner",
    "Returned To Owner":   "Return to Owner",
    "Redemption (Offsite)": "Return to Owner",
    "Euthanized":          "Euthanasia",
    "Euthanasia - RTO":    "Euthanasia",
}

# 舊資料的 outcome_type -> 5 類 outcome_category（與 01_cleaning 一致）
OUTCOME_CATEGORY_MAP = {
    "Adoption":        "adopted",
    "Rto-Adopt":       "adopted",
    "Return to Owner": "return to owner",
    "Transfer":        "transfer",
    "Euthanasia":      "died",
    "Disposal":        "died",
    "Died":            "died",
}

# 決策（intake_reason 對映）：source_name(21 類) -> 舊 intake_reason 詞彙
#   舊詞彙只有 6 種：Stray / Owner Surrender / Public Assist / Abandoned /
#                    Euthanasia Request / Wildlife
#   '# 確定' = 語意明確；'# 推測' = 判斷性對映，可調整。
#   沒列到的 source_name -> 保留原字串；OHE 用 handle_unknown='ignore' 會自動當成
#   零向量（中性 baseline），不會出錯。
SOURCE_NAME_TO_INTAKE_REASON = {
    "Stray":                              "Stray",            # 確定
    "ACO - Impound":                      "Stray",            # 推測：ACO 攔截多為走失/流浪
    "Owner Surrender":                    "Owner Surrender",  # 確定
    "ACO - Owner Surrender":              "Owner Surrender",  # 確定
    "Ambulance - Owner Surrender":        "Owner Surrender",  # 確定
    "Humane Officer Surrendered":         "Owner Surrender",  # 確定
    "Returns":                            "Owner Surrender",  # 推測：認養退回 ≈ 棄養
    "Abandoned":                          "Abandoned",        # 確定
    "Community Animal - Program":         "Public Assist",    # 推測：社區動物計畫
    "Community Animal - Field Services":  "Public Assist",    # 推測
    "Community Animal - General Public":  "Public Assist",    # 推測
    "Agency":                             "Public Assist",    # 推測
    "Ambulance":                          "Public Assist",    # 推測
    "Humane Officer":                     "Public Assist",    # 推測
    "Shelter Offspring":                  "Public Assist",    # 推測：院內出生，舊詞彙無對應
    "Foster Offspring":                   "Public Assist",    # 推測
    "Cruelty Case Offspring":             "Public Assist",    # 推測
    "Wildlife":                           "Wildlife",         # 確定
    "Ambulance - Wildlife":               "Wildlife",         # 確定
    "ACO - Euthanasia Request":           "Euthanasia Request",  # 確定
    "Ambulance - Euthanasia Request":     "Euthanasia Request",  # 確定
}

# intake_health_condition：新資料寫法 -> 舊資料詞彙（HEALTH_KEEP 以外的反正會被
# 03_modeling 折成 'Other'，這裡只做明顯同義字的正規化）
HEALTH_VALUE_MAP = {
    "Neonate":  "Neonatal",
    "NEONATAL": "Neonatal",
    "MEDICAL":  "Medical",
    "MED ATTN": "Med Attn",
    "Normal":   "Normal",
    "Injured":  "Injured",
    "Sick":     "Sick",
    "Dead":     "Dead",
}
# 新資料 intake_health_condition 有 ~83% 為 NaN（舊資料 86% 是 'Normal'）。
# 預設保留 NaN（下游 03_modeling 會折成 'Other'）。若認定「空白=Normal」可改成 "Normal"。
HEALTH_NA_FILL = None

# is_mix：★ 新資料完全缺失，無欄位也無 breed 字串可解析。
# 模型的 LR passthrough 需要非空數值欄，預設填 0（=視為純種）。這是「無資料下的假設」，
# 會與訓練資料（混種佔多數）分布不同 —— 警告會在輸出印出。
IS_MIX_FILL = 0


def banner(t):
    print("\n" + "=" * 78 + "\n" + t + "\n" + "=" * 78)

In [11]:
# ===========================================================================
# 載入 + 日期解析
# ===========================================================================
intakes = pd.read_csv(INTAKES_PATH)
outcomes = pd.read_csv(OUTCOMES_PATH)

intakes["source_date"] = pd.to_datetime(intakes["source_date"], errors="coerce")
intakes["date_of_birth"] = pd.to_datetime(intakes["date_of_birth"], errors="coerce")
outcomes["outcome_date"] = pd.to_datetime(outcomes["outcome_date"], errors="coerce")

banner("[0] 輸入")
print(f"intakes : {intakes.shape}")
print(f"outcomes: {outcomes.shape}")


[0] 輸入
intakes : (14132, 15)
outcomes: (13068, 16)


In [12]:
# ===========================================================================
# [1] merge_asof：每筆 outcome 配到「緊鄰其前、同一隻動物」的 intake
# ===========================================================================
banner("[1] merge_asof 對齊配對（backward, by=animal_id）")

intake_cols = [
    "animal_id", "source_date", "source_name", "type", "sex", "primary_breed",
    "primary_color", "secondary_color", "intake_health_condition",
    "ispreviouslyspayedneutered", "date_of_birth", "found_address", "name_at_intake",
]
intakes_sorted = (
    intakes.dropna(subset=["source_date"])
    .sort_values("source_date")[intake_cols]
    .rename(columns={c: f"i_{c}" for c in intake_cols if c not in ("animal_id", "source_date")})
)
outcomes_sorted = outcomes.dropna(subset=["outcome_date"]).sort_values("outcome_date")

merged = pd.merge_asof(
    outcomes_sorted,
    intakes_sorted,
    left_on="outcome_date",
    right_on="source_date",
    by="animal_id",
    direction="backward",
)

n_total = len(merged)
n_matched = merged["source_date"].notna().sum()
print(f"outcome 總數      : {n_total}")
print(f"配到 intake       : {n_matched}  ({n_matched / n_total:.1%})")
print(f"配不到 intake     : {n_total - n_matched}  ({(n_total - n_matched) / n_total:.1%})")

if DROP_UNMATCHED:
    merged = merged[merged["source_date"].notna()].copy()
    print(f"決策 2：排除配不到 intake 的列 -> 剩 {len(merged)} 列")


[1] merge_asof 對齊配對（backward, by=animal_id）
outcome 總數      : 13068
配到 intake       : 12128  (92.8%)
配不到 intake     : 940  (7.2%)
決策 2：排除配不到 intake 的列 -> 剩 12128 列


In [13]:
# ===========================================================================
# [2] 決策 1：row-level 母體過濾（依 outcome_status）
# ===========================================================================
banner("[2] 母體過濾：保留正常收容流程的 outcome_status")
keep_mask = merged["outcome_status"].isin(OUTCOME_STATUS_TO_TYPE.keys())
dropped = merged[~keep_mask]
print("被排除（非母體）的 outcome_status 分布：")
print(dropped["outcome_status"].value_counts(dropna=False).to_string())
merged = merged[keep_mask].copy()
print(f"\n保留後列數: {len(merged)}")


[2] 母體過濾：保留正常收容流程的 outcome_status
被排除（非母體）的 outcome_status 分布：
outcome_status
Unassisted Death                 282
Return to Habitat                159
DOA                              152
Unassisted Death - In Transit     52
Unassisted Death - In Foster      22
DOA - Released to Owner           11
Unresolved File                    6
Escaped                            1
Stolen                             1
Bite Quarantine (Home)             1

保留後列數: 11441


In [14]:
# ===========================================================================
# [3] 限定 Dog / Cat（Kitten->Cat, Puppy->Dog）
# ===========================================================================
banner("[3] 限定 animal_type ∈ {Dog, Cat}")
type_norm = merged["i_type"].replace({"Kitten": "Cat", "Puppy": "Dog"})
merged["animal_type"] = type_norm
before = len(merged)
merged = merged[merged["animal_type"].isin(["Dog", "Cat"])].copy()
print(f"非貓狗剔除 {before - len(merged)} 列 -> 剩 {len(merged)} 列")
print(merged["animal_type"].value_counts().to_string())


[3] 限定 animal_type ∈ {Dog, Cat}
非貓狗剔除 504 列 -> 剩 10937 列
animal_type
Cat    6471
Dog    4466


In [15]:
# ===========================================================================
# [4] 建構特徵欄（對齊 df_full_merged 訓練 schema）
# ===========================================================================
banner("[4] 建構欄位（重算 length_of_stay，套用各對映表）")

out = pd.DataFrame(index=merged.index)
out["animal_id"] = merged["animal_id"]
out["name"] = merged["name"]
out["animal_type"] = merged["animal_type"]
out["sex"] = merged["i_sex"].fillna("Unknown")
out["primary_breed"] = merged["i_primary_breed"]            # 新資料已預先拆好
out["is_mix"] = IS_MIX_FILL                                  # ★ 完全缺失，填預設

out["date_of_birth"] = merged["i_date_of_birth"]
out["primary_color"] = merged["i_primary_color"]
out["pattern"] = pd.NA                                       # 新資料無 pattern（非模型特徵）
out["secondary_color"] = merged["i_secondary_color"]

# 真正的本次進所時間（重算 LOS 的基準）
out["intake_date"] = merged["source_date"]

# intake_reason 對映（未列到的保留原字串，OHE 會 ignore）
out["intake_reason"] = merged["i_source_name"].map(SOURCE_NAME_TO_INTAKE_REASON).fillna(
    merged["i_source_name"]
)
out["found_location"] = merged["i_found_address"]

# age_at_intake_days = intake_date - date_of_birth
out["age_at_intake_days"] = (out["intake_date"] - out["date_of_birth"]).dt.days

# intake_health_condition 正規化
hc = merged["i_intake_health_condition"].map(HEALTH_VALUE_MAP).fillna(
    merged["i_intake_health_condition"]
)
if HEALTH_NA_FILL is not None:
    hc = hc.fillna(HEALTH_NA_FILL)
out["intake_health_condition"] = hc

# is_previously_spayed_neutered：intake 端布林 -> 1.0/0.0（與訓練 float 一致）
out["is_previously_spayed_neutered"] = (
    merged["i_ispreviouslyspayedneutered"].map({True: 1.0, False: 0.0})
)

out["outcome_date"] = merged["outcome_date"]
out["outcome_type"] = merged["outcome_status"].map(OUTCOME_STATUS_TO_TYPE)
out["outcome_subtype"] = merged["euthanasia_reason"]        # 最接近的 subtype 來源

# ★ 重算單次住院天數（絕不用原檔 days_in_shelter）
out["length_of_stay_days"] = (out["outcome_date"] - out["intake_date"]).dt.days
out["is_long_stay"] = (out["length_of_stay_days"] > LONG_STAY_THRESHOLD).astype(int)
out["is_adopted"] = out["outcome_type"].isin(["Adoption", "Rto-Adopt"]).astype(int)
out["outcome_category"] = out["outcome_type"].map(OUTCOME_CATEGORY_MAP).fillna("other")

out["intake_year"] = out["intake_date"].dt.year
out["intake_month"] = out["intake_date"].dt.month
out["intake_weekday"] = out["intake_date"].dt.weekday


[4] 建構欄位（重算 length_of_stay，套用各對映表）


In [16]:
# ===========================================================================
# [5] 收尾過濾：負天數 / 負年齡（對齊訓練清理）
# ===========================================================================
banner("[5] 收尾過濾")
before = len(out)
out = out[out["length_of_stay_days"] >= 0].copy()
print(f"length_of_stay_days < 0 剔除 {before - len(out)} 列")
before = len(out)
out = out[~(out["age_at_intake_days"] < 0)].copy()
print(f"age_at_intake_days < 0 剔除 {before - len(out)} 列")
out["age_at_intake_days"] = out["age_at_intake_days"].astype("Int64")
before = len(out)
out = out.drop_duplicates()
print(f"drop_duplicates 剔除 {before - len(out)} 列")


[5] 收尾過濾
length_of_stay_days < 0 剔除 0 列
age_at_intake_days < 0 剔除 50 列
drop_duplicates 剔除 0 列


In [17]:
# ===========================================================================
# [6] 對齊欄位順序 + 寫檔
# ===========================================================================
banner("[6] 對齊 df_full_merged schema 並輸出")
train_cols = list(pd.read_csv(TRAIN_SCHEMA_PATH, nrows=0).columns)
missing = [c for c in train_cols if c not in out.columns]
extra = [c for c in out.columns if c not in train_cols]
print(f"訓練 schema 欄位數: {len(train_cols)}")
print(f"本檔缺少的欄位    : {missing if missing else '無'}")
print(f"本檔多出的欄位    : {extra if extra else '無'}")
out = out[train_cols]   # 嚴格對齊順序；若有 missing 會在此 raise，提醒修正

out.to_csv(OUTPUT_PATH, index=False)
print(f"\n已寫出: {OUTPUT_PATH}   shape={out.shape}")


[6] 對齊 df_full_merged schema 並輸出
訓練 schema 欄位數: 26
本檔缺少的欄位    : 無
本檔多出的欄位    : 無

已寫出: ../../data/processed/oot_test_set.csv   shape=(10887, 26)


In [18]:
# ===========================================================================
# [7] 健全度檢查 + 判讀提醒
# ===========================================================================
banner("[7] 健全度檢查（對照訓練：Dog ~0.16 / Cat ~0.25）")
print("重算後 is_long_stay 率 by animal_type：")
print(out.groupby("animal_type")["is_long_stay"].agg(["mean", "count"]).to_string())

print("\nlength_of_stay_days describe()：")
print(out["length_of_stay_days"].describe().to_string())

print("\nintake_reason 分布（對映後；非舊 6 詞彙者 = OHE 會 ignore）：")
print(out["intake_reason"].value_counts(dropna=False).to_string())

unmapped = sorted(set(merged["i_source_name"]) - set(SOURCE_NAME_TO_INTAKE_REASON))
print(f"\n未對映到舊詞彙的 source_name（OHE 會視為零向量）: {unmapped if unmapped else '無'}")

print("\noutcome_category 分布：")
print(out["outcome_category"].value_counts(dropna=False).to_string())

print("\nintake_health_condition 分布（NaN 下游折成 'Other'）：")
print(out["intake_health_condition"].value_counts(dropna=False).to_string())

banner("⚠ 判讀提醒（這些是無資料下的假設，可在腳本頂端常數調整）")
print(f"1. is_mix 完全缺失 -> 全部填 {IS_MIX_FILL}（IS_MIX_FILL）。"
      f"訓練資料混種佔多數，分布會偏移，LR 對此特徵的貢獻在本 test set 失真。")
print(f"2. intake_health_condition ~{out['intake_health_condition'].isna().mean():.0%} 為 NaN"
      f"（HEALTH_NA_FILL={HEALTH_NA_FILL}）-> 下游折成 'Other'。若認定空白=Normal 請改常數。")
print("3. intake_reason 為 source_name 的判斷性對映（見頂端 '推測' 標記），"
      "尤其 ACO - Impound / Community Animal - * / Returns 影響量大。")
print("4. is_previously_spayed_neutered 取自 intake 布林欄（無 Unknown）；"
      "outcomes 的 spayed_neutered 字串欄（含大小寫不一致的 yes/Yes）未使用。")
print("\n完成。")


[7] 健全度檢查（對照訓練：Dog ~0.16 / Cat ~0.25）
重算後 is_long_stay 率 by animal_type：
                 mean  count
animal_type                 
Cat          0.212574   6426
Dog          0.247254   4461

length_of_stay_days describe()：
count    10887.000000
mean        22.159732
std         36.031376
min          0.000000
25%          3.000000
50%          8.000000
75%         28.000000
max        376.000000

intake_reason 分布（對映後；非舊 6 詞彙者 = OHE 會 ignore）：
intake_reason
Stray              6312
Public Assist      3076
Owner Surrender    1426
Abandoned            73

未對映到舊詞彙的 source_name（OHE 會視為零向量）: 無

outcome_category 分布：
outcome_category
adopted            5560
transfer           3727
return to owner    1081
died                519

intake_health_condition 分布（NaN 下游折成 'Other'）：
intake_health_condition
NaN         9162
Normal       980
Injured      325
Sick         259
Neonatal     120
Dead          34
Med Attn       4
Medical        3

⚠ 判讀提醒（這些是無資料下的假設，可在腳本頂端常數調整）
1. is_mix 完全缺失 -> 全部填 0（IS_MIX_FI